## Сравнение типов ControlNet — коллаж для выбора
Запускается поверх текущих ячеек 1-5, использует их `DRIVE_ROOT`, `FONT_PATH` и `render_runes`.  
Генерирует одну фразу через все 6 типов ControlNet (SD 1.5, 512×512), собирает коллаж и сохраняет на Drive.

In [ ]:
# ── ЯЧЕЙКА A. Препроцессоры control-изображений (только cv2/numpy) ─────
import numpy as np, cv2
from PIL import Image

def preproc_canny(img, low=50, high=150):
    """Жёсткие бинарные контуры — текущий выбор."""
    gray=np.array(img.convert('L')); e=cv2.Canny(gray,low,high)
    e=cv2.dilate(e,np.ones((2,2),np.uint8),1)
    return Image.fromarray(np.stack([e]*3,2))

def preproc_hed(img):
    """HED-аппроксимация: размытые мягкие края (gaussian по бинарной маске)."""
    gray=np.array(img.convert('L'))
    _,m=cv2.threshold(gray,200,255,cv2.THRESH_BINARY_INV)
    s=cv2.GaussianBlur(m.astype(np.float32),(15,15),3)
    s=(s/s.max()*255).astype(np.uint8) if s.max()>0 else s.astype(np.uint8)
    return Image.fromarray(np.stack([s]*3,2))

def preproc_scribble(img):
    """Scribble: толстые неровные штрихи на белом фоне."""
    gray=np.array(img.convert('L'))
    _,m=cv2.threshold(gray,200,255,cv2.THRESH_BINARY_INV)
    thick=cv2.dilate(m,np.ones((5,5),np.uint8),2)
    rng=np.random.default_rng(42)
    noise=(rng.uniform(0,0.4,thick.shape)*thick).astype(np.uint8)
    sc=np.clip(thick.astype(int)+noise,0,255).astype(np.uint8)
    return Image.fromarray(255-np.stack([sc]*3,2))   # белый фон

def preproc_lineart(img):
    """Lineart: утончённые одиночные линии на белом фоне."""
    gray=np.array(img.convert('L'))
    _,m=cv2.threshold(gray,200,255,cv2.THRESH_BINARY_INV)
    thin=cv2.erode(m,np.ones((3,3),np.uint8),1)
    return Image.fromarray(255-np.stack([thin]*3,2))  # белый фон

def preproc_mlsd(img):
    """MLSD: только прямые отрезки (HoughLinesP) — теряет кривые."""
    gray=np.array(img.convert('L')); e=cv2.Canny(gray,50,150)
    canvas=np.zeros_like(e)
    lines=cv2.HoughLinesP(e,1,np.pi/180,threshold=15,minLineLength=10,maxLineGap=8)
    if lines is not None:
        for l in lines:
            x1,y1,x2,y2=l[0]; cv2.line(canvas,(x1,y1),(x2,y2),255,2)
    return Image.fromarray(np.stack([canvas]*3,2))

def preproc_depth(img):
    """Карта глубины: фон=светлый (близко), руны=тёмные (глубина канавки)."""
    gray=np.array(img.convert('L'))
    _,m=cv2.threshold(gray,200,255,cv2.THRESH_BINARY_INV)
    depth=np.full(gray.shape,200,dtype=np.uint8); depth[m>0]=70
    depth=cv2.GaussianBlur(depth,(9,9),4)
    return Image.fromarray(np.stack([depth]*3,2))

PREPROCS = [
    ('Canny',          preproc_canny,    'выбрана: жёсткий контур'),
    ('HED / soft-edge',preproc_hed,      'мягкие края → дрейф'),
    ('Scribble',       preproc_scribble, 'очень свободный → дрейф'),
    ('Lineart',        preproc_lineart,  'тонкая линия, несов. домен'),
    ('MLSD',           preproc_mlsd,     'только прямые, теряет кривые'),
    ('Depth',          preproc_depth,    'плоский рендер → нет глубины'),
]
print('Препроцессоры готовы:', [n for n,_,_ in PREPROCS])

In [ ]:
# ── ЯЧЕЙКА B. Генерация 6×SD1.5+ControlNet → коллаж на Drive ────────────
import gc, torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

# SD1.5 — все типы ControlNet хорошо поддерживаются (lllyasviel v1.0/v1.1)
BASE   = 'runwayml/stable-diffusion-v1-5'
SZ     = 512
SEED   = 42
PROMPT = ("ancient runic inscription carved into dark grey granite, "
          "deep incised runes, weathered rough eroded surface, monochrome grey, "
          "archaeological macro photo, sharp focus, realistic stone texture")
NEG    = "smooth polished, fantasy, glow, blur, watermark, bright colors, illustration"

CN_IDS = {
    'Canny':           'lllyasviel/sd-controlnet-canny',
    'HED / soft-edge': 'lllyasviel/control_v11p_sd15_softedge',
    'Scribble':        'lllyasviel/control_v11p_sd15_scribble',
    'Lineart':         'lllyasviel/control_v11p_sd15_lineart',
    'MLSD':            'lllyasviel/sd-controlnet-mlsd',
    'Depth':           'lllyasviel/control_v11p_sd15_depth',
}

# Фраза для сравнения (короткая, 3 слова — репрезентативна для всех типов)
TEXT = 'ᛅᚢᚴ᛬ᚠᛅᚦᚢᚱ᛬ᛋᛏᛁᚾ'   # auk᛬faþur᛬stin
base_img = render_runes(TEXT, canvas=SZ, req_size=80)   # из ячейки 3

results = {}
for name, preproc_fn, note in PREPROCS:
    ctrl = preproc_fn(base_img)
    cn_id = CN_IDS.get(name)
    print(f'\n▶ {name}  [{cn_id}]')
    try:
        cn = ControlNetModel.from_pretrained(cn_id, torch_dtype=torch.float16, token=HF_TOKEN)
        pipe = StableDiffusionControlNetPipeline.from_pretrained(
            BASE, controlnet=cn, torch_dtype=torch.float16,
            safety_checker=None, token=HF_TOKEN)
        pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
        pipe.enable_model_cpu_offload()
        g = torch.Generator('cuda').manual_seed(SEED)
        gen = pipe(PROMPT, negative_prompt=NEG, image=ctrl,
                   num_inference_steps=25, guidance_scale=7.0,
                   controlnet_conditioning_scale=0.8,
                   generator=g, width=SZ, height=SZ).images[0]
        results[name] = (ctrl, gen, note); print('  ✓ готово')
    except Exception as e:
        print(f'  ✗ {e}')
        results[name] = (ctrl, None, note)
    finally:
        try: del pipe, cn
        except: pass
        gc.collect(); torch.cuda.empty_cache()

# ── Сборка коллажа ──────────────────────────────────────────────────────
from PIL import ImageDraw, ImageFont

CTRL_H=110; CELL=512; LABEL_H=52; PAD=10; COLS=3
CELL_TOTAL_H = CTRL_H + CELL + LABEL_H
W = COLS*(CELL+PAD)+PAD; H = 2*(CELL_TOTAL_H+PAD)+PAD+55
collage = Image.new('RGB',(W,H),(20,20,20)); d=ImageDraw.Draw(collage)
try:
    fb=ImageFont.truetype('/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',20)
    fn=ImageFont.truetype('/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',14)
    ft=ImageFont.truetype('/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',17)
except:
    fb=fn=ft=ImageFont.load_default()

d.text((PAD,PAD),'ControlNet comparison — SD 1.5 512×512  |  text: auk᛬faþur᛬stin  |  seed=42',
       fill=(210,210,210),font=fn)

for idx,(name,_,_) in enumerate(PREPROCS):
    col=idx%COLS; row=idx//COLS
    x0=PAD+col*(CELL+PAD); y0=55+PAD+row*(CELL_TOTAL_H+PAD)
    ctrl,gen,note = results.get(name,(None,None,''))
    # control image strip
    if ctrl:
        cs=ctrl.resize((CELL,CTRL_H),Image.LANCZOS)
        collage.paste(cs,(x0,y0))
    d.rectangle([x0,y0,x0+CELL-1,y0+CTRL_H-1],outline=(80,80,80))
    d.text((x0+5,y0+3),'control image',fill=(180,180,180),font=fn)
    # generated image
    gy=y0+CTRL_H
    if gen:
        collage.paste(gen.resize((CELL,CELL),Image.LANCZOS),(x0,gy))
    else:
        err=Image.new('RGB',(CELL,CELL),(50,20,20)); de=ImageDraw.Draw(err)
        de.text((CELL//2-30,CELL//2-10),'ERROR',fill=(255,80,80),font=fb)
        collage.paste(err,(x0,gy))
    d.rectangle([x0,gy,x0+CELL-1,gy+CELL-1],outline=(80,80,80))
    # label
    ly=gy+CELL+4; col_hi=(255,220,80) if 'Canny' in name else (180,180,180)
    d.text((x0+5,ly),name,fill=col_hi,font=ft)
    d.text((x0+5,ly+22),note,fill=(140,140,140),font=fn)

out='/content/controlnet_comparison.png'
collage.save(out)
# копия на Drive
try:
    import shutil; shutil.copy2(out, str(DRIVE_ROOT/'controlnet_comparison.png'))
    print(f'Сохранено: {out}  +  Drive')
except Exception as e:
    print(f'Сохранено: {out}  (Drive: {e})')

from IPython.display import display, Image as IpyImg
display(IpyImg(filename=out,width=900))